# 02. Optimizers, schedules and other training choices (XLM-R base)

With the data fixed by `01_data_mix` (`ner/recipes.py`), we change one thing at a time against the
baseline: AdamW, lr 3e-5, linear decay with 10% warmup, weight decay 0.01, 3 epochs, batch 32, 256-token
windows, all sub-tokens labeled. Every run uses validation fold 0, so the scores are comparable with each
other and with notebook 01.

| Group | What changes | Why it might matter |
| --- | --- | --- |
| learning rate | 1e-5, 5e-5, 1e-4 | too low underfits in 3 epochs; too high can wreck the pre-trained weights |
| schedule | cosine, constant after warmup, no warmup | how fast the step size shrinks; warmup protects the pre-trained weights from the random head's first large gradients |
| optimizer | Adafactor, SGD with momentum | Adafactor keeps factored second moments (less memory); SGD has no per-parameter scaling, and is known to fine-tune transformers poorly |
| layer-wise lr decay | 0.9 per layer | lower layers hold general language knowledge and should change less |
| regularization | label smoothing 0.1, frozen embeddings | the labels are noisy (pseudo-labels, LLM labels, annotator style); frozen embeddings keep rare place names' pre-trained vectors |
| labels | first sub-token only | the original BERT NER recipe; fewer training targets, but no conflicting targets inside a word |
| shape | 128-token windows, batch 16, 5 epochs | shorter context and more steps |

For the report: loss and F1 curves of each group below, gradient-flow heatmaps for the most different
optimizers, and all runs in W&B (project `dl-a4-ner`, tag `02_optimization`).

### Setup

On Colab (T4 or better): clones the repo, installs W&B and keeps every run on Drive in
`dl-a4-ner/runs`, so a disconnect loses at most the run in progress (finished runs are loaded, not
retrained). Teammates: put a shortcut to the shared `dl-a4-ner` folder in your My Drive, so all runs
land in one place. Set `WANDB_ENTITY` to the team's W&B entity to log into one shared project.
Locally the notebook runs from the repo root and writes to `transformers/runs`. To check the whole
notebook in a few minutes before using GPU time, set the environment variable `NER_SMOKE=1`: every run
shrinks to a few hundred texts and goes to `transformers/runs_smoke`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/koalla05/deep-learning-assignment4.git"
WANDB_PROJECT = "dl-a4-ner"  # None switches W&B off
WANDB_ENTITY = None  # the team's W&B entity, so all teammates log into one project

if "google.colab" in sys.modules:
    repo = Path("/content/deep-learning-assignment4")
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    from google.colab import drive

    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/dl-a4-ner/runs")
else:
    if Path.cwd().name == "transformers":
        os.chdir("..")
    OUT = Path("transformers/runs")
sys.path.insert(0, os.getcwd())
if os.environ.get("NER_SMOKE"):  # quick end-to-end check with tiny runs, see ner.train.smoke
    OUT, WANDB_PROJECT = Path("transformers/runs_smoke"), None
OUT.mkdir(parents=True, exist_ok=True)
if WANDB_ENTITY:
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY
if WANDB_PROJECT:
    import wandb

    wandb.login()

In [ ]:
import dataclasses

from ner import plots, recipes
from ner.train import RunConfig, load_run, run_or_load

BASE = RunConfig(
    name="base", model="FacebookAI/xlm-roberta-base", fold=0, data=recipes.DATA,
    **(recipes.STAGE2 or {}), **recipes.OPTIMIZATION, tags=("02_optimization",),
)
print(BASE)


def train(name, **changes):
    cfg = dataclasses.replace(BASE, name=f"02-{name}", **changes)
    return run_or_load(cfg, OUT, WANDB_PROJECT)

## Baseline

The same settings as the winning run of notebook 01, trained again under this notebook's name, so every
comparison below differs from it in exactly one setting.

In [ ]:
train("baseline")

## Learning rate

In [ ]:
train("lr1e-5", learning_rate=1e-5)

In [ ]:
train("lr5e-5", learning_rate=5e-5)

In [ ]:
train("lr1e-4", learning_rate=1e-4)

## Schedule

In [ ]:
train("cosine", scheduler="cosine")

In [ ]:
train("constant", scheduler="constant_with_warmup")

In [ ]:
train("no-warmup", warmup_ratio=0.0)

## Optimizer

Each optimizer gets the learning rate it usually needs: Adafactor without relative steps behaves like
Adam with a slightly larger lr; SGD needs a much larger one because it doesn't divide by the gradient's
running scale.

In [ ]:
train("adafactor", optimizer="adafactor", learning_rate=5e-5)

In [ ]:
train("sgd", optimizer="sgd", learning_rate=5e-3)

## Layer-wise learning-rate decay

Each layer gets 0.9 times the learning rate of the layer above it: the head 5e-5, the top layer 4.5e-5,
the bottom layer about 1.4e-5 and the embeddings about 1.3e-5.

In [ ]:
train("llrd0.9", layer_lr_decay=0.9, learning_rate=5e-5)

## Regularization and labels

In [ ]:
train("smooth0.1", label_smoothing=0.1)

In [ ]:
train("freeze-emb", freeze_embeddings=True)

In [ ]:
train("first-subtoken", label_all_subtokens=False)

## Shape of the training

In [ ]:
train("len128", max_length=128, stride=32)

In [ ]:
train("batch16", batch_size=16)

In [ ]:
train("epochs5", epochs=5)

## Results

In [ ]:
GROUPS = {
    "learning rate": ["baseline", "lr1e-5", "lr5e-5", "lr1e-4"],
    "schedule": ["baseline", "cosine", "constant", "no-warmup"],
    "optimizer": ["baseline", "adafactor", "sgd", "llrd0.9"],
    "regularization and labels": ["baseline", "smooth0.1", "freeze-emb", "first-subtoken"],
    "shape": ["baseline", "len128", "batch16", "epochs5"],
}
names = dict.fromkeys(n for group in GROUPS.values() for n in group)
runs = {n: load_run(OUT, f"02-{n}") for n in names if (OUT / f"02-{n}" / "metrics.json").exists()}
plots.results_table(runs, leaderboard="transformers/leaderboard.csv").round(4)

In [ ]:
for title, group in GROUPS.items():
    plots.plot_curves({n: runs[n] for n in group if n in runs}, title=title)

Gradient flow: with AdamW the update size doesn't depend on the gradient size, so small gradients in
the lower layers are not a problem by themselves. With SGD they are: the update is the gradient times
the learning rate, so layers with tiny gradients barely move. Compare the two heatmaps.

In [ ]:
for n in ["baseline", "sgd", "llrd0.9", "lr1e-4"]:
    if n in runs:
        plots.plot_grad_flow(runs[n], title=n)

## Conclusions

Pick the setting per group by `half_B`, keeping in mind that one run per setting has seed noise (in
Assignment 3 a seed change moved CV by about 0.013). Changes smaller than that are ties: keep the
simpler option. Write the result into `ner/recipes.py` (`OPTIMIZATION`) for the model notebooks.